# AI-Based Lightweight Mobile Crop Disease Detection using MobileViT Small

This notebook trains a lightweight **MobileViT Small** model using PyTorch and the `timm` library to classify tomato crop leaf diseases. It is specifically designed to run on **Google Colab** and integrates automated dataset path detection, Google Drive unzipping, transfer learning, evaluation metrics, and final model exports.

In [ ]:
# Install required libraries
!pip install timm tqdm scikit-learn matplotlib seaborn

## 1. Google Drive Mounting & Dataset Extraction
This cell mounts Google Drive and automatically extracts your `archive.zip` dataset into `/content/dataset/` if it hasn't been extracted yet. It also auto-detects the directories containing `train/` and `valid/` folders dynamically.

In [ ]:
import os
import zipfile
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path definitions
ZIP_PATH = '/content/drive/MyDrive/archive.zip'
EXTRACT_DIR = '/content/dataset/'

# 3. Check and unzip dataset
if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)

# Check if train and valid exist in extraction dir (or a nested archive dir)
def find_dataset_root(root_dir):
    for root, dirs, files in os.walk(root_dir):
        if 'train' in dirs and 'valid' in dirs:
            return root
    return None

dataset_root = find_dataset_root(EXTRACT_DIR)

if dataset_root is None:
    print(f"Dataset not found or not extracted. Unzipping {ZIP_PATH} to {EXTRACT_DIR}...")
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(f"Source ZIP file not found at {ZIP_PATH}. Please make sure your dataset is named 'archive.zip' and uploaded to the root of your Google Drive.")
    
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        from tqdm import tqdm
        for file in tqdm(zip_ref.infolist(), desc="Extracting dataset"):
            zip_ref.extract(file, EXTRACT_DIR)
            
    dataset_root = find_dataset_root(EXTRACT_DIR)
    if dataset_root is None:
        raise ValueError("Could not find 'train' and 'valid' directories after extraction. Please check your ZIP file structure.")
else:
    print(f"Dataset already extracted.")

print(f"Detected dataset root folder: {dataset_root}")
TRAIN_DIR = os.path.join(dataset_root, 'train')
VALID_DIR = os.path.join(dataset_root, 'valid')

## 2. Transforms & DataLoaders Setup
We specify MobileViT standard 224x224 input sizing, apply data augmentations to prevent overfitting (Random Horizontal/Vertical Flips, Rotation, Jitter, Affine), and load the datasets dynamically. It will automatically detect all subfolder class names.

In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Image size config
IMAGE_SIZE = 224

# Data Processing
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load Datasets dynamically
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
valid_dataset = datasets.ImageFolder(VALID_DIR, transform=valid_transform)

# Extract and print classes
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Detected {num_classes} classes: {class_names}")

# Data Dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 3. Pretrained MobileViT Small Architecture
We load MobileViT Small (`mobilevit_s`) from `timm` with ImageNet weights, and replace the head classifier dynamically to match our dataset classes.

In [ ]:
import timm
import torch.nn as nn

# Load pretrained MobileViT Small
print("Initializing MobileViT Small model...")
model = timm.create_model('mobilevit_s', pretrained=True)

# Replace head dynamically to match our dataset classes
num_features = model.head.fc.in_features
model.head.fc = nn.Linear(num_features, num_classes)
model = model.to(device)

print(f"Model initialized. Head classifier updated to output {num_classes} categories.")

## 4. Helper Callbacks (Freezing Backbone & Early Stopping)
To optimize transfer learning, we implement:
- `set_backbone_freeze`: Freezes layers for the first 5 epochs to train only the classifier head.
- `EarlyStopping`: Standard callback class with patience=5 to prevent overfitting.

In [ ]:
import time
import json
import numpy as np

# Helper functions for freezing backbone
def set_backbone_freeze(model, freeze=True):
    # Freezes or unfreezes all layers except the classifier head (model.head)
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = not freeze
    print(f"MobileViT Backbone status: {'FROZEN' if freeze else 'UNFROZEN (Fine-Tuning)'}")

# Early Stopping Class
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

## 5. Training Loop
We train using `AdamW`, `CosineAnnealingLR`, and `CrossEntropyLoss` for 20 epochs. The first 5 epochs freeze the backbone. At epoch 6, we unfreeze the backbone for full fine-tuning.

In [ ]:
# Optimizer & Scheduler Setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)
EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
early_stopping = EarlyStopping(patience=5)

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "lr": []
}

best_val_acc = 0.0

# Freeze backbone for first 5 epochs
set_backbone_freeze(model, freeze=True)

for epoch in range(1, EPOCHS + 1):
    # Unfreeze at epoch 6 (index 5)
    if epoch == 6:
        set_backbone_freeze(model, freeze=False)
        # Reinitialize optimizer to include all parameters
        optimizer = torch.optim.AdamW(model.parameters(), lr=scheduler.get_last_lr()[0])
        # Reinitialize scheduler from current step
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - 5)

    start_time = time.time()
    
    # ------------------ TRAINING ------------------
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    from tqdm import tqdm
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        train_bar.set_postfix(loss=loss.item())
        
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = correct_train / total_train
    
    # ------------------ VALIDATION ------------------
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    val_bar = tqdm(valid_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]")
    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
            val_bar.set_postfix(loss=loss.item())
            
    val_loss = running_val_loss / len(valid_loader.dataset)
    val_acc = correct_val / total_val
    
    current_lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - start_time
    
    # Step scheduler
    scheduler.step()
    
    # Record history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)
    
    # Print metrics
    print(f"\n--- Epoch {epoch} Summary ---")
    print(f"Training Loss:     {train_loss:.4f}")
    print(f"Validation Loss:   {val_loss:.4f}")
    print(f"Training Acc:      {train_acc * 100:.2f}%")
    print(f"Validation Acc:    {val_acc * 100:.2f}%")
    print(f"Learning Rate:     {current_lr:.6f}")
    print(f"Training Time:     {epoch_time:.2f}s")
    print("-" * 25)
    
    # Save Best Model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"⭐ New best validation accuracy: {val_acc*100:.2f}%. Saved best_model.pth")
        
    # Save Last Model
    torch.save(model.state_dict(), 'last_model.pth')
    
    # Early Stopping check
    early_stopping(val_loss)
    if early_stopping.counter >= early_stopping.patience:
        print("Early stopping triggered. Training stopped.")
        break

# Save class names & history
with open('class_names.json', 'w') as f:
    json.dump(class_names, f, indent=4)
    
with open('training_history.json', 'w') as f:
    json.dump(history, f, indent=4)

## 6. Evaluation Curves & Confusion Matrix
This cell evaluates the model's performance on the validation dataset using the saved `best_model.pth` and generates confusion matrix heatmaps and accuracy/loss curves.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# Load best model for evaluation
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Metrics
overall_acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

print("\n================ EVALUATION METRICS ================")
print(f"Overall Accuracy: {overall_acc * 100:.2f}%")
print(f"Precision (Macro): {precision * 100:.2f}%")
print(f"Recall (Macro):    {recall * 100:.2f}%")
print(f"F1-Score (Macro):  {f1 * 100:.2f}%")
print("====================================================")

# Classification Report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

# Plots
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label='Training Loss', color='blue')
plt.plot(epochs_range, history["val_loss"], label='Validation Loss', color='red')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label='Training Accuracy', color='blue')
plt.plot(epochs_range, history["val_acc"], label='Validation Accuracy', color='red')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('learning_curves.png')
plt.show()